In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

*사용 예상 시간: Eagle 프로세서에서 약 1분 (참고: 이는 예상치이며, 실제 실행 시간은 다를 수 있습니다.)*

## 배경

Circuit-knitting은 회로를 게이트 수 및/또는 Qubit 수가 더 적은 여러 개의 작은 서브회로로 분할하는 다양한 방법을 포괄하는 용어입니다. 각 서브회로는 독립적으로 실행될 수 있으며, 최종 결과는 각 서브회로의 실행 결과에 대한 고전 후처리를 통해 얻습니다. 이 기법은 [circuit cutting Qiskit addon](https://qiskit.github.io/qiskit-addon-cutting/index.html)을 통해 사용할 수 있으며, 기법에 대한 자세한 설명은 [문서](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html)와 기타 [입문 자료](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html)에서 확인할 수 있습니다.

이 노트북은 회로를 와이어 방향으로 분할하는 <b>와이어 커팅(wire cutting)</b>이라는 방법을 다룹니다 [\[1\], \[2\]](#references). 고전 회로에서는 분할 지점의 상태가 결정론적으로 0 또는 1로 확정되므로 분할이 간단하지만, 양자 회로에서는 절단 지점에서 Qubit의 상태가 일반적으로 혼합 상태입니다. 따라서 각 서브회로는 서로 다른 기저(보통 Pauli 기저 [\[3\], \[4\]](#references)와 같은 토모그래피적으로 완전한 기저 집합)로 여러 번 측정되어야 하며, 그에 대응하는 고유 상태로 준비되어야 합니다. 아래 그림(<i>출처: Ritajit Majumdar의 박사 학위 논문</i>)은 4-Qubit GHZ 상태를 세 개의 서브회로로 와이어 커팅하는 예시를 보여줍니다. 여기서 $M_j$는 기저 집합(보통 Pauli X, Y, Z)을 나타내고, $P_i$는 고유 상태 집합(보통 $|0\rangle$, $|1\rangle$, $|+\rangle$, $|+i\rangle$)을 나타냅니다.

![wc-1.png](../docs/images/tutorials/wire-cutting-to-improve-performance/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting-to-improve-performance/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

각 서브회로는 Qubit 수 및/또는 게이트 수가 더 적기 때문에 노이즈의 영향을 덜 받을 것으로 예상됩니다. 이 노트북은 이 방법을 사용하여 시스템의 노이즈를 효과적으로 억제하는 예시를 보여줍니다.

## 요구 사항

이 튜토리얼을 시작하기 전에 다음이 설치되어 있는지 확인하세요:

- [시각화](https://docs.quantum.ibm.com/api/qiskit/visualization) 지원이 포함된 Qiskit SDK v2.0 이상
- Qiskit Runtime v0.22 이상 ( `pip install qiskit-ibm-runtime` )
- Circuit cutting Qiskit addon v0.9.0 이상 (`pip install qiskit-addon-cutting`)

이 노트북에서는 MBL(Many Body Localization) 회로를 사용합니다. MBL 회로는 하드웨어 효율적인 회로로, $\theta$와 $\vec{\phi}$ 두 가지 파라미터로 매개변수화됩니다. $\theta$를 $0$으로 설정하고 모든 Qubit의 초기 상태를 $|0\rangle$으로 준비하면, $\vec{\phi}$의 값에 무관하게 모든 Qubit 위치 $i$에 대해 $\langle Z_i \rangle$의 이상적인 기댓값은 $+1$입니다. MBL 회로에 대한 자세한 내용은 <a href="https://arxiv.org/abs/2307.07552">이 논문</a>을 참고하세요.

## 설정

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## Part I. 소규모 예시

### 1단계: 고전 입력을 양자 문제로 변환

먼저 특정 파라미터 값 없이 템플릿 Circuit을 구성합니다. 절단 위치를 표시하기 위해 `CutWire`라는 플레이스홀더도 삽입합니다. 소규모 예시로 10-Qubit MBL 회로를 사용합니다.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

We calculate the average expectation value $O = \frac{1}{n} \sum_i Z_i$ over all qubits for $\theta = 0$. Since the ideal expectation value of $\langle Z_i \rangle = 1$ $\forall$ $i$, the ideal expectation value of $O$ is also $1$. The parameters $\phi$ are selected randomly.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

이제 Circuit에 **CutWire**를 삽입하여 절단 위치를 표시합니다. 원래 Circuit의 Qubit 수를 $n$이라 할 때, $\frac{n}{2}$ 번째 Qubit 이후에 절단이 이루어지도록 두 개의 대략 동일한 절단을 만듭니다. 함수에서 `use_cut=True`로 설정합니다.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/31844134-514b-46ea-85f9-133e432f053f-0.avif)

### 2단계: 양자 하드웨어 실행을 위한 문제 최적화
다음으로 Circuit을 두 개의 작은 서브회로로 절단합니다. 이 예시에서는 2개의 서브회로만 사용합니다. 이를 위해 <a href="https://qiskit.github.io/qiskit-addon-cutting/">Qiskit Addon: Circuit Cutting</a>을 사용합니다.
#### 회로를 더 작은 서브회로로 절단
와이어를 한 지점에서 절단하면 Qubit 수가 하나 증가합니다. 원래 Qubit 외에, 절단 이후의 Circuit에 플레이스홀더로 추가 Qubit이 생깁니다. 다음 그림이 이를 나타냅니다:

![wc-4.png](../docs/images/tutorials/wire-cutting-to-improve-performance/dfc5f923-e507-4873-888e-d90e1618be3a.avif)

이 Addon은 `cut_wires` 함수를 사용하여 절단으로 인해 추가되는 Qubit을 처리합니다.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

#### 관측량 생성 및 확장
이제 관측량 $M_z = \frac{1}{n}\sum_{i=1}^n \langle Z_i \rangle$를 구성합니다. 각 $i$에 대해 $\langle Z_i \rangle$의 이상적인 값이 $+1$이므로, $M_z$의 이상적인 값도 $+1$입니다.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

단, 절단 이후 가상의 2-Qubit `Move` 연산이 삽입되면서 Circuit의 Qubit 수가 증가했습니다. 따라서 현재 Circuit에 맞추어 관측량도 항등 연산자를 삽입하여 확장해야 합니다.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

서브회로를 시각화합니다.

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

#### Transpile the circuits onto the backend

For the first example involving only simulation, we transpile the circuit into the basis gate set of the backend:

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/35920640-76e8-4af6-a252-ee6a22e9c26a-0.avif)

관측량도 서브회로에 맞게 분할되었습니다.

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

각 서브회로는 여러 개의 샘플을 생성합니다. 재구성 시 이 모든 샘플의 결과가 반영됩니다. 각 샘플을 `subexperiment`라고 합니다.
`Move` 연산을 사용하여 관측량을 확장하려면 `PauliList` 데이터 구조가 필요합니다. 서브실험 재구성 시에도 유용하게 쓸 수 있도록 $M_z$ 관측량을 보다 범용적인 `SparsePauliOp` 데이터 구조로도 생성할 수 있습니다.

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

절단된 Qubit이 서로 다른 기저로 측정되는 두 가지 예시를 살펴봅니다. 첫 번째는 일반적인 Z 기저로, 두 번째는 X 기저로 측정됩니다.

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/987547e4-296a-41e4-ad82-41f4139a87a0-0.avif)

#### 각 부분 실험 트랜스파일하기

현재는 회로를 실행하기 전에 트랜스파일해야 합니다. 따라서 먼저 각 부분 실험의 회로를 트랜스파일합니다.